In [15]:
"""
trend_ranking_and_correlations.py
==================================

Two purely descriptive, data-driven analyses on monthly_trend.csv --
no composite indices, no invented theory.

1. TREND RANKING
   For each of the four BAT constructs (EX, EMO, COG, MD), compute:
     - first-year mean vs. last-full-year mean, and the % change between them
     - HAC-OLS slope per year (with p-value)
     - Theil-Sen robust slope per year
   then rank the four constructs by % change so you can directly answer
   "which construct grew the most / least."

2. CORRELATION WITH VOLUME AND SEVERITY
   For each construct's monthly rate, compute the Pearson correlation
   (with p-value) against:
     - n_posts          (subreddit activity/volume that month)
     - burnout_rate      (independently-computed high-burnout classification rate)
     - bat_score_mean    (the composite severity score)

   IMPORTANT CAVEAT, stated up front so it isn't missed:
   bat_score_mean is *arithmetically* the sum of the four *_yes_rate
   columns (verified: bat_score_mean = EX_rate + EMO_rate + COG_rate +
   MD_rate, ratio ~4.000 to avg_yes_rate). So correlating a construct
   against bat_score_mean is partially circular by construction -- a
   high correlation there mostly just reflects "this construct is one
   of the four things being summed," not an independent empirical
   relationship. It's reported for completeness/transparency, but
   correlation with n_posts and burnout_rate are the more meaningful,
   non-circular numbers here.

Usage:
    python3 trend_ranking_and_correlations.py monthly_trend.csv ./outputs
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

CONSTRUCTS = ["EX", "EMO", "COG", "MD"]
RATE_COLS = {c: f"{c}_yes_rate" for c in CONSTRUCTS}
CONSTRUCT_LABELS = {
    "EX": "Exhaustion",
    "EMO": "Emotional Impairment",
    "COG": "Cognitive Impairment",
    "MD": "Mental Distance",
}
CONSTRUCT_COLORS = {
    "EX": "#1f77b4",
    "EMO": "#ff7f0e",
    "COG": "#2ca02c",
    "MD": "#d62728",
}
HAC_MAXLAGS = 12


def load_data(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["year_month"], format="%Y-%m")
    df = df.sort_values("date").reset_index(drop=True)
    df["t_month"] = np.arange(len(df))
    df["t_year"] = df["t_month"] / 12.0
    df["year"] = df["date"].dt.year
    return df


def hac_ols_slope(y, x_years):
    X = sm.add_constant(np.asarray(x_years, dtype=float))
    res = sm.OLS(np.asarray(y, dtype=float), X).fit(
        cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS}
    )
    return res.params[1], res.pvalues[1]


def theil_sen_slope(y, x_years):
    slope, _, _, _ = stats.theilslopes(np.asarray(y, dtype=float),
                                        np.asarray(x_years, dtype=float))
    return slope


# ----------------------------------------------------------------------
# 1. Trend ranking
# ----------------------------------------------------------------------

def build_trend_ranking(df, out_dir):
    first_year = int(df["year"].min())
    last_full_year = int(df[df["year"] < df["year"].max()]["year"].max())

    rows = []
    for c in CONSTRUCTS:
        col = RATE_COLS[c]
        first_val = df[df["year"] == first_year][col].mean()
        last_val = df[df["year"] == last_full_year][col].mean()
        pct_change = 100 * (last_val - first_val) / first_val

        hac_slope, hac_p = hac_ols_slope(df[col].values, df["t_year"].values)
        ts_slope = theil_sen_slope(df[col].values, df["t_year"].values)

        rows.append({
            "construct": c,
            "label": CONSTRUCT_LABELS[c],
            f"{first_year}_mean": first_val,
            f"{last_full_year}_mean": last_val,
            "pct_change": pct_change,
            "hac_slope_per_year": hac_slope,
            "hac_p_value": hac_p,
            "theilsen_slope_per_year": ts_slope,
        })

    table = pd.DataFrame(rows).sort_values("pct_change", ascending=False).reset_index(drop=True)
    table.insert(0, "rank", range(1, len(table) + 1))
    table.to_csv(out_dir / "trend_ranking.csv", index=False)
    return table, first_year, last_full_year


def plot_trend_ranking(table, first_year, last_full_year, out_dir):
    fig, ax = plt.subplots(figsize=(10, 7))
    ordered = table.sort_values("pct_change", ascending=True)
    colors = [CONSTRUCT_COLORS[c] for c in ordered["construct"]]
    bars = ax.barh(ordered["label"], ordered["pct_change"], color=colors)

    for bar, val in zip(bars, ordered["pct_change"]):
        ax.text(val + (1 if val >= 0 else -1), bar.get_y() + bar.get_height() / 2,
                 f"{val:+.1f}%", va="center",
                 ha="left" if val >= 0 else "right", fontsize=10)

    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_xlabel(f"% change in annual average rate, {first_year} mean vs {last_full_year} mean")
    ax.set_title("Which BAT Category grew the most?")
    fig.tight_layout()
    fig.savefig(out_dir / "fig_trend_ranking.png", dpi=150)
    plt.close(fig)


# ----------------------------------------------------------------------
# 2. Correlation with volume and severity
# ----------------------------------------------------------------------
 
def build_correlations(df, out_dir):
    rows = []
    for c in CONSTRUCTS:
        col = RATE_COLS[c]
        r_vol, p_vol = stats.pearsonr(df[col], df["n_posts"])
        r_rate, p_rate = stats.pearsonr(df[col], df["burnout_rate"])
        r_bat, p_bat = stats.pearsonr(df[col], df["bat_score_mean"])
        rows.append({
            "construct": c,
            "label": CONSTRUCT_LABELS[c],
            "corr_with_n_posts": r_vol,
            "p_n_posts": p_vol,
            "corr_with_burnout_rate": r_rate,
            "p_burnout_rate": p_rate,
            "corr_with_bat_score_mean": r_bat,
            "p_bat_score_mean": p_bat,
        })
    table = pd.DataFrame(rows)
    table.to_csv(out_dir / "correlation_with_volume_severity.csv", index=False)
    return table
 
 
def plot_correlations(table, out_dir):
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(CONSTRUCTS))
    width = 0.25
 
    vol = table["corr_with_n_posts"].values
    rate = table["corr_with_burnout_rate"].values
    bat = table["corr_with_bat_score_mean"].values
 
    ax.bar(x - width, vol, width, label="vs. n_posts (volume)", color="#7f7f7f")
    ax.bar(x, rate, width, label="vs. burnout_rate", color="#d62728")
    ax.bar(x + width, bat, width, label="vs. bat_score_mean", color="#aec7e8")
 
    ax.set_xticks(x)
    ax.set_xticklabels(table["construct"])
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_ylabel("Pearson r")
    ax.set_title("Each category's correlation with volume vs. severity measures")
 
    # Legend placed outside the plot area (to the right), but still part of
    # the saved figure -- avoids covering the bars.
    ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1.02, 1.0),
               borderaxespad=0)
 
    # Caveat footnote also placed outside the axes, below the plot.
    fig.text(0.5, 0.01,
              "*bat_score_mean is arithmetically the sum of all four construct rates -- this bar is expected to be high by construction.",
              ha="center", fontsize=8, style="italic", color="gray")
 
    fig.tight_layout(rect=[0, 0.04, 1, 1])
    fig.savefig(out_dir / "fig_correlations.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
 
 

# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------

def main():
    in_path =  "monthly_trend.csv"
    out_dir = Path(sys.argv[2]) if len(sys.argv) > 2 else Path("./outputs")
    out_dir.mkdir(parents=True, exist_ok=True)

    df = load_data(in_path)

    ranking, first_year, last_full_year = build_trend_ranking(df, out_dir)
    plot_trend_ranking(ranking, first_year, last_full_year, out_dir)

    corr_table = build_correlations(df, out_dir)
    plot_correlations(corr_table, out_dir)

    print("=" * 70)
    print(f"TREND RANKING ({first_year} -> {last_full_year})")
    print("=" * 70)
    print(ranking.to_string(index=False))

    print()
    print("=" * 70)
    print("CORRELATION WITH VOLUME AND SEVERITY")
    print("(bat_score_mean column is partially circular -- see script docstring)")
    print("=" * 70)
    print(corr_table.to_string(index=False))

    print(f"\nSaved tables and figures to: {out_dir.resolve()}")


if __name__ == "__main__":
    main()

TREND RANKING (2018 -> 2025)
 rank construct                label  2018_mean  2025_mean  pct_change  hac_slope_per_year  hac_p_value  theilsen_slope_per_year
    1        MD      Mental Distance   0.040475   0.062992   55.631048            0.004180 8.844138e-12                 0.004200
    2       COG Cognitive Impairment   0.026075   0.034900   33.844679            0.001170 1.133701e-05                 0.001200
    3        EX           Exhaustion   0.058550   0.075158   28.366069            0.003000 2.208491e-12                 0.002995
    4       EMO Emotional Impairment   0.156125   0.173108   10.878036            0.003845 5.919185e-05                 0.003693

CORRELATION WITH VOLUME AND SEVERITY
(bat_score_mean column is partially circular -- see script docstring)
construct                label  corr_with_n_posts    p_n_posts  corr_with_burnout_rate  p_burnout_rate  corr_with_bat_score_mean  p_bat_score_mean
       EX           Exhaustion           0.519299 3.098404e-08         